# 5 · Call types: windows, evaluation, discovery and a frozen classifier library

Build a call-type library from whistle contours, and use it to classify new recordings.

**How it works**
1. **Contours**: whistle contours (e.g. from PAMGuard's Whistle and Moan Detector) are clipped to a frequency band, filtered, and measured (duration, frequencies, slopes, inflections...).
2. **Windows**: contours are grouped into short sliding windows (default 4 s, 50 % overlap). Each window is summarised by its contours: means and spreads of the measurements, the longest contour's shape, how much of the window is tonal, and a frequency profile.
3. **Labels**: windows take labels from call annotations (call type, plus any other columns such as species or population).
4. **Evaluate**: how well do the labelled call types separate? (random forest, cross-validated by day)
5. **Discover**: cluster all windows, labelled or not (importance-weighted cosine distance + an outlier check). Clusters are described by the call types in them, and judged for **binary label tasks** you define (e.g. species X vs Y): a cluster gets a verdict only if its share of a label is reliably above or below the overall rate, on enough days. Leave-one-day-out tests show how well the verdicts generalise, per window and over longer decision blocks.
6. **Library**: everything needed to classify new data the same way is frozen into one JSON file, including optional **decision rules** you define (e.g. "blocks whose contours barely vary in frequency are noise").
7. **Classify**: new contours → windows → nearest cluster → verdicts → clock-aligned blocks decided by majority vote → rules. `standalone.py` does the same with numpy only, to run anywhere.

By default it runs on a small **synthetic** example: 4 days of contours (`examples/contours_example_synthetic.json.gz`), with call annotations for the first 3 (`examples/call_annotations_example_synthetic.csv`). Day 4 plays the role of new data, and includes a stretch of tonal noise.

## Setup
Locally: `pip install -e "python[notebooks,pamguard]"` from the repository folder. In Google Colab, run the next cell.

In [ ]:
# Google Colab only
# %pip install -q "speciesclassifiers[pamguard] @ git+https://github.com/tristankleyn/speciesClassifiers#subdirectory=python"

In [ ]:
import gzip
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from speciesclassifiers.calltypes import (ContourParams, DiscoverParams, EvalParams, LabelTask, ReliabilityParams,
                                          WindowParams, build_library, classify_contours, discover, discovery_pool,
                                          evaluate, label_windows, load_library, make_windows, prepare_contours,
                                          save_library, select_rows, self_check, window_features)
from speciesclassifiers.calltypes import standalone
from speciesclassifiers.io import read_annotations, read_whistles
from speciesclassifiers.io.annotation_tables import read_annotation_table
from speciesclassifiers.plots import plot_confusion, plot_importance

## 1 · Load contours and call annotations
**Contours**, choose one source:
- `"example"`: the synthetic example (a JSON list of contours, the format `standalone.py` reads).
- `"binaries"`: PAMGuard Whistle and Moan Detector binary files (all under a folder). The detector's sample rate and FFT settings are worked out from the files.

**Call annotations**, one row per call with `event_id, label, start, end` (UTC) plus any other label columns (e.g. `species`, `ecotype`). Either:
- `"csv"`: a CSV in this format, or
- `"tables"`: annotation tables (one per recording / day; text or Excel), converted with `read_annotation_table`.

Calls of no particular type can be labelled `untyped`; they still count for the other label columns.

In [ ]:
CONTOUR_SOURCE = "example"        # "example" or "binaries"
EXAMPLE_CONTOURS = "../../examples/contours_example_synthetic.json.gz"
BINARY_FOLDER = "path/to/binaries"

ANNOTATION_SOURCE = "csv"         # "csv" or "tables"
ANNOTATIONS_CSV = "../../examples/call_annotations_example_synthetic.csv"
ANNOTATION_TABLES = ["path/to/table1.txt"]
TABLE_OPTIONS = dict(label_col="Annotation", keep_cols=("Ecotype",))   # see read_annotation_table

OTHER_LABELS = ["species", "ecotype"]    # annotation columns copied to windows, besides the call type ("label")
OUTPUT_DIR = "calltypes_run"

out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
if CONTOUR_SOURCE == "example":
    with gzip.open(EXAMPLE_CONTOURS, "rt") as f:
        whistles = pd.DataFrame(json.load(f))
else:
    whistles, detector = read_whistles(BINARY_FOLDER)
    print("Detector settings:", detector)
if ANNOTATION_SOURCE == "csv":
    calls = read_annotations(ANNOTATIONS_CSV)
else:
    calls = read_annotations(read_annotation_table(ANNOTATION_TABLES, **TABLE_OPTIONS))
annotated_days = sorted(pd.to_datetime(calls["start"], unit="s", utc=True).dt.strftime("%Y%m%d").unique())
print(f"{len(whistles)} contours; {len(calls)} annotated calls on {len(annotated_days)} days: {annotated_days}")
calls["label"].value_counts()

## 2 · Contours and windows
**Contour filters** (`ContourParams`): frequency band (points outside are clipped off), minimum duration, optional minimum SNR.

**Windows** (`WindowParams`): length and overlap, how contours are assigned (`"overlap"` or `"midpoint"`), filters (minimum contours, occupancy, longest contour), and the number of frequency-profile bins.

In [ ]:
contour_p = ContourParams(fmin_hz=800, fmax_hz=16000, min_contour_s=0.4, min_snr_db=None)
window_p = WindowParams(window_s=4, overlap=0.5, assign_by="overlap", min_contours=2, min_occupancy=0,
                        min_longest_contour_s=0, freq_profile_bins=32,
                        fmin_hz=contour_p.fmin_hz, fmax_hz=contour_p.fmax_hz)

contours, points = prepare_contours(whistles, contour_p)
windows = window_features(make_windows(contours, window_p), contours, points, window_p)
print(f"{len(contours)} contours kept -> {len(windows)} windows")

## 3 · Labels
Each window takes the label of the call it overlaps most, among calls it covers at least half of. Windows that only clip a call are `partial`, windows with several call types are `mixed`, others `none`. `OTHER_LABELS` columns are copied from the same call.

In [ ]:
windows = label_windows(windows, calls, window_p, min_call_coverage=0.5, carry=OTHER_LABELS)
windows.to_csv(out / "windows.csv", index=False)
pd.crosstab(windows["label"], windows["day"])

## 4 · Evaluate: how well do the labelled call types separate?
Labelled windows (capped per call type and day) → random forest, cross-validated by day (or by time block when there are too few days). `skill` is balanced accuracy scaled so that 0 = chance and 1 = perfect.

In [ ]:
eval_p = EvalParams(max_per_class_per_day=150, min_class_n=8, rf_trees=300,
                    exclude_features=["amplitude", "snr", "signal", "noise", "mean_width"])
res = evaluate(select_rows(windows, eval_p), eval_p)
print({k: res[k] for k in ["n_windows", "n_classes", "cv_grouping", "rf_bal_acc", "skill"]})
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
plot_confusion(res["confusion"], "Call types (cross-validated)", ax=ax[0])
plot_importance(res["feature_importance"], n=12, ax=ax[1])
plt.tight_layout()

## 5 · Discover
All windows go into the **discovery pool**: labelled ones, `untyped` ones, and (optionally) windows from days without annotations. Excluded call-type labels (`none`, `partial`, `mixed`...) count as unlabelled.

**Label tasks** are binary questions each cluster is judged on: `LabelTask(name, column, positive, negative)`. A cluster's verdict is `positive` / `negative` / `uncertain`.

Main settings:
- `DiscoverParams`: `cosine_threshold` (larger = fewer, broader clusters), `min_cluster_size`, `euclid_outlier_q` (outlier check), `clusterer` (`"agglomerative"` or `"hdbscan"`).
- `ReliabilityParams`: `prior_strength` (shrinkage of small clusters), `ci` (credible interval), `min_label_days`, `decision_windows` (block lengths to test).

In [ ]:
tasks = [LabelTask("species", "species", "X", "Y"),
         LabelTask("ecotype", "ecotype", "E1", "E2")]
discover_p = DiscoverParams(clusterer="agglomerative", cosine_threshold=0.7, min_cluster_size=10,
                            euclid_outlier_q=0.95, new_min_n=10)
reliability_p = ReliabilityParams(cap_per_day=50, prior_strength=10, ci=0.9, min_label_days=2)

pool = discovery_pool(windows, annotated_days, include_unannotated_days=True)
result = discover(pool, discover_p, tasks, reliability_p)
result["summary"].to_csv(out / "clusters.csv", index=False)
cols = ["cluster", "n", "n_days", "top_call_type", "call_types", "candidate_new"] + [f"{t.name}_verdict" for t in tasks]
result["summary"][cols]

**Scorecard**: number of clusters, share unassigned, silhouette, agreement with labels (AMI), and per task the leave-one-day-out coverage (share of windows that get a verdict), balanced accuracy and skill. The decision-block table shows how much combining windows over longer blocks helps.

In [ ]:
pd.Series(result["scorecard"]).round(3)

In [ ]:
d = result["decision_windows"]
d[d["class"] == "overall"].pivot(index="decision_window", columns="task", values=["coverage", "accuracy_when_predicted", "skill"]).reindex(
    ["window", *reliability_p.decision_windows]).round(2)

## 6 · Library
Freeze the clusters, verdicts and feature transform into one JSON file.

- `BLOCK_MINUTES`: decisions are made per clock-aligned block of this length (majority vote of the windows that get a verdict).
- `REQUIRES`: report a task only when another task's decision is a given value, e.g. ecotype only in species-X blocks (otherwise `n/a`).
- `RULES`: optional decision rules, checked in order after voting; the first that matches sets every task's decision to its `"then"` value. A rule compares one block value with a number (`<`, `<=`, `>`, `>=`, `==`, `!=`). Block values: `n_windows`, `n_assigned`, `n_contours`, `block_meanfreq_range_hz` (range of the contours' mean frequencies), `block_median_abs_slope_hz_s` (median contour wobble), `block_median_freq_hz`, `block_median_contour_s`. No rules by default; the example rule marks blocks whose contours barely vary in frequency as noise.

In [ ]:
BLOCK_MINUTES = 5
REQUIRES = {"ecotype": ("species", "X")}
RULES = [
    {"if": "block_meanfreq_range_hz", "op": "<", "value": 200, "then": "noise"},
]

library = build_library(result, contour_p, window_p, tasks, block_minutes=BLOCK_MINUTES,
                        rules=RULES, requires=REQUIRES)
save_library(library, out / "library.json")
print(f"{library['library_version']}: {len(library['clusters'])} clusters, "
      f"{len(library['feature_spec']['cols'])} features, {(out / 'library.json').stat().st_size / 1e6:.1f} MB")
print("Re-assigning the discover windows with the frozen library:", self_check(result, library))

## 7 · Classify new data
New contours (here: the example's day 4) → windows → clusters → verdicts → decision blocks. `period=(start, end)` (Unix s) would also list blocks without detections (`no_detections`).

`raw_<task>` is the vote before rules; `rule` says which rule fired.

In [ ]:
library = load_library(out / "library.json")
new_start = pd.Timestamp("2024-06-04", tz="UTC").timestamp()
new = whistles[whistles["start"] >= new_start]

new_windows, blocks, _ = classify_contours(new, library)
blocks.to_csv(out / "blocks.csv", index=False)
show = blocks.round(2).assign(block_start=pd.to_datetime(blocks["block_start"], unit="s", utc=True))
show[["block_start", "n_windows", "n_assigned", "species", "ecotype", "rule", "raw_species",
      "species_vote_share", "block_meanfreq_range_hz"]]

### The standalone classifier
`standalone.py` (in `speciesclassifiers/calltypes/`) needs only numpy: copy it and `library.json` to wherever classification runs. It reads contours as a list of dicts (`start`, `times`, `freqs`, `slice_s`, optional `snr_db`; `contour_from_bins` converts peak bins and slice numbers) and gives the same windows and blocks as above.

From the command line: `python standalone.py library.json contours.json blocks.csv [windows.csv]`

In [ ]:
lib = standalone.load_library(out / "library.json")
w2, b2 = standalone.classify(new.to_dict("records"), lib)
b2 = pd.DataFrame(b2)
print("Same block decisions as the package:",
      all((b2[t["name"]].tolist() == blocks[t["name"]].tolist()) for t in lib["tasks"]))